In [1]:
import scanpy as sc
import omicverse as ov
import pandas as pd
ov.plot_set()


   ____            _     _    __                  
  / __ \____ ___  (_)___| |  / /__  _____________ 
 / / / / __ `__ \/ / ___/ | / / _ \/ ___/ ___/ _ \ 
/ /_/ / / / / / / / /__ | |/ /  __/ /  (__  )  __/ 
\____/_/ /_/ /_/_/\___/ |___/\___/_/  /____/\___/                                              

Version: 1.6.11, Tutorials: https://omicverse.readthedocs.io/
Dependency error: The 'phate>=1.0' distribution was not found and is required by the application


In [2]:
adata = sc.read("/home/lugli/spuccio/Projects/SP039/GBmap/Wang2019_Part2.h5ad")

In [3]:
adata = adata[adata.obs['donor_id'].isin(["SF11644", "SF11956", "SF11979", "SF10022", "SF10127", "SF4297", "SF6996", "SF9259R", "SF9259S"])]

In [4]:
df_obs = pd.DataFrame(adata.obs)

In [5]:
del adata.obs

In [6]:
adata = adata.raw.to_adata()

In [7]:
adata

AnnData object with n_obs × n_vars = 22611 × 16828
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'highly_variable_rank', 'highly_variable_features'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [10]:
#adata = adata.raw.to_adata()

In [11]:
X_counts_recovered, size_factors_sub=ov.pp.recover_counts(adata.X, 50*1e4, 50*1e5, log_base=None, 
                                                          chunk_size=10000)


100%|██████████| 2611/2611 [00:03<00:00, 790.64it/s]


In [12]:
adata.X = X_counts_recovered

In [13]:
annot = sc.queries.biomart_annotations(
    "hsapiens",
    ["external_gene_name","ensembl_gene_id", "start_position", "end_position", "chromosome_name",],
).set_index("external_gene_name")

In [14]:
annot

,ensembl_gene_id,start_position,end_position,chromosome_name
external_gene_name,,,,
MT-TF,ENSG00000210049,577,647,MT
MT-RNR1,ENSG00000211459,648,1601,MT
MT-TV,ENSG00000210077,1602,1670,MT
MT-RNR2,ENSG00000210082,1671,3229,MT
MT-TL1,ENSG00000209082,3230,3304,MT
...,...,...,...,...
SCMH1-DT,ENSG00000235358,41241772,41338644,1
LINC01740,ENSG00000228067,212467563,212556085,1
SLC44A3-AS1,ENSG00000293271,94585556,94855426,1


In [15]:
adata.var.columns

Index(['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances', 'highly_variable_rank',
       'highly_variable_features'],
      dtype='object')

In [16]:
adata.var = adata.var[['mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances',
       'residual_variances']]

In [17]:
adata.var 

,mt,n_cells,percent_cells,robust,means,variances,residual_variances
feature_name,,,,,,,
ZNF470-DT,False,1019,4.506656,True,0.021429,0.012549,0.550056
AC092667.2,False,240,1.061430,True,0.003879,0.001666,0.387455
ZNF367,False,667,2.949892,True,0.014200,0.009108,0.562397
HDHD2,False,4039,17.862987,True,0.103092,0.066965,0.736628
MORF4L2-AS1,False,274,1.211800,True,0.004929,0.002344,0.468810
...,...,...,...,...,...,...,...
NOP53-AS1,False,342,1.512538,True,0.007161,0.004306,0.570292
CFAP20DC-AS1,False,192,0.849144,True,0.004189,0.002653,0.592118
MIR4713HG,False,250,1.105657,True,0.007379,0.006578,0.877543


In [18]:
df_tmp = pd.merge(adata.var , annot, left_index=True, right_index=True, how='left')

In [19]:
df_tmp = df_tmp.reset_index().drop_duplicates(['feature_name']).set_index(['feature_name'])

In [20]:
adata.var = df_tmp

In [21]:
adata = adata[:,adata.var['chromosome_name'].isin(["1","2","3","4","5","6","7","8","9","10","11","12","13","14","15","16","17","18","19","20","21","22","X","Y","MT"])]

In [22]:
adata

View of AnnData object with n_obs × n_vars = 22611 × 14454
    var: 'mt', 'n_cells', 'percent_cells', 'robust', 'means', 'variances', 'residual_variances', 'ensembl_gene_id', 'start_position', 'end_position', 'chromosome_name'
    uns: 'X_approximate_distribution', 'annotation_level_1_colors', 'annotation_level_2_colors', 'annotation_level_3_colors', 'batch_condition', 'default_embedding', 'donor_id_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'rank_genes_groups', 'scaled|original|cum_sum_eigenvalues', 'scaled|original|pca_var_ratios', 'schema_version', 'scsa_celltype_cellmarker_colors', 'scsa_celltype_panglaodb_colors', 'title', 'umap'
    obsm: 'X_harmony', 'X_pca', 'X_umap', 'scaled|original|X_pca'
    obsp: 'connectivities', 'distances'

In [23]:
adata.obs['donor_id'] = df_obs['donor_id']

In [24]:
metadata_data = {
    'Author': ['Wang2019'] * 9,
    'donor_id': ["SF11644", "SF11956", "SF11979", "SF10022", "SF10127", "SF4297", "SF6996", "SF9259R", "SF9259S"],
    'stage': ['Primary'] * 9,
    'assay': ['10x 3\' v2'] * 9,
    'tissue': ['brain'] * 9,
    'Cells': ['Total'] * 9,
    'Method': ['cell', 'cell', 'cell', 'nuclei', 'nuclei', 'nuclei', 'nuclei', 'nuclei', 'nuclei']
}

metadata_df = pd.DataFrame(metadata_data)

# Display the metadata DataFrame
print(metadata_df)

     Author donor_id    stage      assay tissue  Cells  Method
0  Wang2019  SF11644  Primary  10x 3' v2  brain  Total    cell
1  Wang2019  SF11956  Primary  10x 3' v2  brain  Total    cell
2  Wang2019  SF11979  Primary  10x 3' v2  brain  Total    cell
3  Wang2019  SF10022  Primary  10x 3' v2  brain  Total  nuclei
4  Wang2019  SF10127  Primary  10x 3' v2  brain  Total  nuclei
5  Wang2019   SF4297  Primary  10x 3' v2  brain  Total  nuclei
6  Wang2019   SF6996  Primary  10x 3' v2  brain  Total  nuclei
7  Wang2019  SF9259R  Primary  10x 3' v2  brain  Total  nuclei
8  Wang2019  SF9259S  Primary  10x 3' v2  brain  Total  nuclei


In [25]:
merged_obs_df = pd.merge(pd.DataFrame(adata.obs), metadata_df, left_on='donor_id', right_on='donor_id', how='left')

# Display the merged dataframe
print(merged_obs_df)

      donor_id    Author    stage      assay tissue  Cells  Method
0      SF11644  Wang2019  Primary  10x 3' v2  brain  Total    cell
1      SF11644  Wang2019  Primary  10x 3' v2  brain  Total    cell
2      SF11644  Wang2019  Primary  10x 3' v2  brain  Total    cell
3      SF11644  Wang2019  Primary  10x 3' v2  brain  Total    cell
4      SF11644  Wang2019  Primary  10x 3' v2  brain  Total    cell
...        ...       ...      ...        ...    ...    ...     ...
22606  SF9259S  Wang2019  Primary  10x 3' v2  brain  Total  nuclei
22607  SF9259S  Wang2019  Primary  10x 3' v2  brain  Total  nuclei
22608  SF9259S  Wang2019  Primary  10x 3' v2  brain  Total  nuclei
22609  SF9259S  Wang2019  Primary  10x 3' v2  brain  Total  nuclei
22610  SF9259S  Wang2019  Primary  10x 3' v2  brain  Total  nuclei

[22611 rows x 7 columns]


In [26]:
df_obs = df_obs[['donor_id','n_genes','nUMIs','annotation_level_1', 'annotation_level_2','annotation_level_3','scsa_celltype_cellmarker', 'scsa_celltype_panglaodb','cell_type']]

In [27]:
df_obs

,donor_id,n_genes,nUMIs,annotation_level_1,annotation_level_2,annotation_level_3,scsa_celltype_cellmarker,scsa_celltype_panglaodb,cell_type
SF11644_AAACCTGAGAACTCGG-0,SF11644,2875,2061.886230,Neoplastic,Stem-like,NPC-like,Astrocyte,Neurons,malignant cell
SF11644_AAACCTGAGTTCCACA-0,SF11644,3395,2187.695557,Neoplastic,Stem-like,NPC-like,Astrocyte,Neurons,malignant cell
SF11644_AAACCTGCAGGAACGT-0,SF11644,4247,2335.208496,Neoplastic,Stem-like,NPC-like,Astrocyte,Endothelial Cells,malignant cell
SF11644_AAACCTGCATCTCGCT-0,SF11644,2230,2077.640137,Neoplastic,Stem-like,NPC-like,Astrocyte,Neurons,malignant cell
SF11644_AAACCTGGTAGGCTGA-0,SF11644,2156,2030.696533,Neoplastic,Stem-like,NPC-like,Astrocyte,Neurons,malignant cell
...,...,...,...,...,...,...,...,...,...
SF9259S_TTTCCTCTCCCTTGGT-1-1,SF9259S,2676,2007.396851,Non-neoplastic,Glial-Neuronal,Oligodendrocyte,Oligodendrocyte,Oligodendrocytes,oligodendrocyte
SF9259S_TTTGGAGTCAACACCA-1-1,SF9259S,1011,1582.634399,Non-neoplastic,Glial-Neuronal,Oligodendrocyte,Oligodendrocyte,Oligodendrocytes,oligodendrocyte
SF9259S_TTTGGTTGTCGTTATG-1-1,SF9259S,1820,1942.878418,Non-neoplastic,Glial-Neuronal,Oligodendrocyte,Oligodendrocyte,Oligodendrocytes,oligodendrocyte
SF9259S_TTTGTTGAGCTCTTCC-1-1,SF9259S,2416,2066.596191,Non-neoplastic,Glial-Neuronal,Oligodendrocyte,Oligodendrocyte,Oligodendrocytes,oligodendrocyte


In [28]:
merged_obs_df.index= df_obs.index

In [29]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method'], dtype='object')

In [30]:
merged_obs_df = pd.merge(merged_obs_df, df_obs,right_index=True,left_index=True, how='left')

In [31]:
merged_obs_df.columns

Index(['donor_id_x', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'donor_id_y', 'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [32]:
del merged_obs_df['donor_id_y']

In [33]:
merged_obs_df.columns = ['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
                         'n_genes', 'nUMIs', 'annotation_level_1',
       'annotation_level_2', 'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type']

In [34]:
merged_obs_df.columns

Index(['donor_id', 'Author', 'stage', 'assay', 'tissue', 'Cells', 'Method',
       'n_genes', 'nUMIs', 'annotation_level_1', 'annotation_level_2',
       'annotation_level_3', 'scsa_celltype_cellmarker',
       'scsa_celltype_panglaodb', 'cell_type'],
      dtype='object')

In [35]:
adata.obs = merged_obs_df

In [36]:
ov.pp.score_genes_cell_cycle(adata,species='human')

calculating cell cycle phase
computing score 'S_score'
    finished: added
    'S_score', score of gene set (adata.obs).
    687 total control genes are used. (0:00:01)
computing score 'G2M_score'
    finished: added
    'G2M_score', score of gene set (adata.obs).
    859 total control genes are used. (0:00:01)
-->     'phase', cell cycle phase (adata.obs)


In [37]:
adata.write("/home/lugli/spuccio/Projects/SP039/GBmap/Wang2019_Part3.h5ad")